# Evaluation Pipeline: KFP on RHOAI Data Science Pipelines

This notebook packages the agent evaluation workflow as a **Kubeflow Pipeline (KFP)**
that runs on Red Hat OpenShift AI Data Science Pipelines. This enables:

- **Scheduled evaluation** — run evaluations on a cadence (daily, weekly)
- **Reproducible runs** — same pipeline, same dataset, versioned results
- **CI/CD integration** — trigger evaluation after model or prompt changes
- **RHOAI-native** — runs inside the OpenShift AI platform with managed resources

## 1. Prerequisites

In [ ]:
import os
import subprocess
from dotenv import load_dotenv

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(env_path)

mlflow_uri = os.getenv("MLFLOW_TRACKING_URI", "")
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "deep-research-harness")
llm_base_url = os.getenv("LLM_BASE_URL", "")
llm_model = os.getenv("LLM_MODEL", "")

checks = [
    ("MLFLOW_TRACKING_URI", bool(mlflow_uri)),
    ("MLFLOW_EXPERIMENT_NAME", bool(experiment_name)),
    ("LLM_BASE_URL", bool(llm_base_url)),
    ("LLM_MODEL", bool(llm_model)),
]
for name, ok in checks:
    status = "\u2705" if ok else "\u26a0\ufe0f Not set"
    print(f"  {status} {name}")

try:
    import kfp
    print(f"\n\u2705 kfp version: {kfp.__version__}")
except ImportError:
    print("\n\u26a0\ufe0f kfp not installed \u2014 pip install kfp")

## 2. Pipeline Architecture

The evaluation pipeline follows a 4-step structure:

```
Step 1: setup_mlflow     →  Configure MLflow tracking and experiment
Step 2: create_dataset   →  Create/load evaluation dataset in MLflow
Step 3: run_evaluation   →  Run scorers (simple or LLM-judge)
Step 4: report_results   →  Generate and display evaluation report
```

Each step runs as an independent container with its own dependencies.
The pipeline supports two modes:
- **Simple** — deterministic scorers only (fast, no LLM calls)
- **LLM-Judge** — full evaluation with LLM-based scorers

## 3. Define Pipeline Components

Each component is a self-contained function that KFP packages into a container.

In [ ]:
from typing import NamedTuple

import kfp
from kfp import dsl
from kfp.dsl import component
from kfp import kubernetes

BASE_IMAGE = "python:3.11-slim"

COMMON_PACKAGES = [
    "mlflow>=2.15.0",
    "python-dotenv>=1.0.0",
    "nest-asyncio>=1.6.0",
    "pydantic>=2.0.0",
    "httpx>=0.27.0",
]

LLM_JUDGE_PACKAGES = COMMON_PACKAGES + [
    "openai>=1.0.0",
]

print("\u2705 KFP imports ready")

### Step 1: Setup MLflow

In [ ]:
@component(base_image=BASE_IMAGE, packages_to_install=COMMON_PACKAGES)
def setup_mlflow_op(
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
    mlflow_workspace: str = "",
) -> str:
    """Configure MLflow tracking and return experiment name."""
    import os
    from pathlib import Path
    import mlflow

    os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"

    if not os.environ.get("MLFLOW_TRACKING_TOKEN"):
        sa_token_path = Path("/var/run/secrets/kubernetes.io/serviceaccount/token")
        if sa_token_path.exists():
            os.environ["MLFLOW_TRACKING_TOKEN"] = sa_token_path.read_text().strip()

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    if mlflow_workspace:
        mlflow.set_workspace(mlflow_workspace)

    experiment_name = mlflow_experiment_name
    if not experiment_name.endswith("-eval"):
        experiment_name = f"{experiment_name}-eval"

    mlflow.set_experiment(experiment_name)
    print(f"MLflow configured: {mlflow_tracking_uri}")
    print(f"Experiment: {experiment_name}")
    return experiment_name


print("\u2705 setup_mlflow_op defined")

### Step 2: Create Dataset

In [ ]:
@component(base_image=BASE_IMAGE, packages_to_install=COMMON_PACKAGES)
def create_dataset_op(
    mlflow_tracking_uri: str,
    experiment_name: str,
    dataset_name: str,
    mlflow_workspace: str = "",
) -> NamedTuple("DatasetOutput", [("experiment_name", str), ("dataset_id", str)]):
    """Create evaluation dataset in MLflow."""
    import os
    from typing import NamedTuple
    from pathlib import Path
    import mlflow
    from mlflow.genai.datasets import create_dataset

    os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"
    if not os.environ.get("MLFLOW_TRACKING_TOKEN"):
        sa_token_path = Path("/var/run/secrets/kubernetes.io/serviceaccount/token")
        if sa_token_path.exists():
            os.environ["MLFLOW_TRACKING_TOKEN"] = sa_token_path.read_text().strip()

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    if mlflow_workspace:
        mlflow.set_workspace(mlflow_workspace)
    mlflow.set_experiment(experiment_name)

    test_cases = [
        {
            "inputs": {"query": "What is the main architecture described in the document?"},
            "expectations": {
                "expected_keywords": ["architecture", "system", "design"],
                "min_length": 200,
                "requires_citations": True,
            },
        },
        {
            "inputs": {"query": "What are the key contributions and innovations?"},
            "expectations": {
                "expected_keywords": ["contribution", "novel"],
                "min_length": 200,
                "requires_citations": True,
            },
        },
        {
            "inputs": {"query": "What evaluation metrics are used and what are the results?"},
            "expectations": {
                "expected_keywords": ["metric", "evaluation", "result"],
                "min_length": 200,
                "requires_citations": True,
            },
        },
    ]

    dataset = create_dataset(
        name=dataset_name,
        tags={"stage": "validation", "version": "1", "domain": "deep-research"},
    )
    dataset = dataset.merge_records(test_cases)
    print(f"Dataset created: {dataset.dataset_id} ({len(test_cases)} cases)")

    DatasetOutput = NamedTuple("DatasetOutput", [("experiment_name", str), ("dataset_id", str)])
    return DatasetOutput(experiment_name=experiment_name, dataset_id=dataset.dataset_id)


print("\u2705 create_dataset_op defined")

### Step 3: Run Evaluation

In [ ]:
@component(base_image=BASE_IMAGE, packages_to_install=LLM_JUDGE_PACKAGES)
def run_eval_op(
    mlflow_tracking_uri: str,
    experiment_name: str,
    dataset_id: str,
    mode: str = "simple",
    llm_base_url: str = "",
    llm_model: str = "",
    mlflow_workspace: str = "",
) -> dict:
    """Run evaluation with configurable mode (simple or llm-judge)."""
    import os
    import re
    import warnings
    from pathlib import Path

    warnings.filterwarnings("ignore")
    os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"

    if not os.environ.get("MLFLOW_TRACKING_TOKEN"):
        sa_token_path = Path("/var/run/secrets/kubernetes.io/serviceaccount/token")
        if sa_token_path.exists():
            os.environ["MLFLOW_TRACKING_TOKEN"] = sa_token_path.read_text().strip()

    if llm_base_url:
        os.environ["OPENAI_API_BASE"] = llm_base_url
        os.environ["OPENAI_BASE_URL"] = llm_base_url

    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        pass

    import mlflow
    from mlflow.genai.scorers import scorer
    from mlflow.genai.datasets import get_dataset

    @scorer
    def contains_expected_keywords(inputs: dict, outputs: str, expectations: dict) -> bool:
        keywords = expectations.get("expected_keywords", [])
        if not keywords:
            return True
        text_lower = str(outputs).lower()
        return any(kw.lower() in text_lower for kw in keywords)

    @scorer
    def has_citations(outputs: str, expectations: dict) -> bool:
        if not expectations.get("requires_citations", False):
            return True
        patterns = [r"\\[\\d+\\]", r"source:", r"according to", r"reference"]
        text = str(outputs).lower()
        return any(re.search(p, text, re.IGNORECASE) for p in patterns)

    @scorer
    def response_depth(outputs: str, expectations: dict) -> float:
        min_length = expectations.get("min_length", 100)
        return 1.0 if len(str(outputs)) >= min_length else 0.5

    def predict_fn(query: str) -> str:
        return f"Mock response for: {query}. This includes architecture, system design, and evaluation results with references [1][2]."

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    if mlflow_workspace:
        mlflow.set_workspace(mlflow_workspace)
    mlflow.set_experiment(experiment_name)

    dataset = get_dataset(dataset_id=dataset_id)
    print(f"Dataset loaded: {dataset.name}")

    scorers = [contains_expected_keywords, has_citations, response_depth]

    if mode == "llm-judge" and llm_base_url and llm_model:
        from mlflow.genai.scorers import Guidelines, RelevanceToQuery, Safety
        judge = f"openai:/{llm_model}"
        scorers.extend([
            RelevanceToQuery(model=judge),
            Safety(model=judge),
            Guidelines(
                name="research_guidelines",
                guidelines=[
                    "Response should address the research query directly",
                    "Response should cite source material",
                    "Response should NOT hallucinate unsupported claims",
                ],
                model=judge,
            ),
        ])

    print(f"Running {mode} evaluation with {len(scorers)} scorers")
    result = mlflow.genai.evaluate(data=dataset, predict_fn=predict_fn, scorers=scorers)

    metrics = {}
    if hasattr(result, "metrics") and result.metrics:
        for k, v in result.metrics.items():
            metrics[k] = round(v, 4) if isinstance(v, float) else v

    print(f"Results: {metrics}")
    return metrics


print("\u2705 run_eval_op defined")

### Step 4: Report Results

In [ ]:
@component(base_image=BASE_IMAGE, packages_to_install=["pydantic>=2.0.0"])
def report_results_op(
    metrics: dict,
    mlflow_tracking_uri: str,
    mode: str,
) -> str:
    """Generate evaluation report."""
    print("=" * 60)
    print(f"EVALUATION REPORT - {mode.upper()} MODE")
    print("=" * 60)
    print("\nMetrics:")
    for metric, value in sorted(metrics.items()):
        if isinstance(value, float):
            print(f"  {metric}: {value:.2%}")
        else:
            print(f"  {metric}: {value}")
    print(f"\nView results: {mlflow_tracking_uri}/#/experiments")
    return f"Evaluation complete ({mode} mode). Metrics: {len(metrics)} recorded."


print("\u2705 report_results_op defined")

## 4. Define Pipelines

Two pipeline variants: one for simple evaluation, one for LLM-as-Judge.

In [ ]:
@dsl.pipeline(
    name="Research Simple Evaluation Pipeline",
    description="Evaluate deep research agent with deterministic scorers",
)
def simple_eval_pipeline(
    mlflow_tracking_uri: str,
    mlflow_workspace: str = "",
    mlflow_experiment_name: str = "deep-research-harness",
    dataset_name: str = "deep_research_eval_simple",
):
    setup_task = setup_mlflow_op(
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
        mlflow_workspace=mlflow_workspace,
    )
    dataset_task = create_dataset_op(
        mlflow_tracking_uri=mlflow_tracking_uri,
        experiment_name=setup_task.output,
        dataset_name=dataset_name,
        mlflow_workspace=mlflow_workspace,
    )
    eval_task = run_eval_op(
        mlflow_tracking_uri=mlflow_tracking_uri,
        experiment_name=dataset_task.outputs["experiment_name"],
        dataset_id=dataset_task.outputs["dataset_id"],
        mode="simple",
        mlflow_workspace=mlflow_workspace,
    )
    report_results_op(
        metrics=eval_task.output,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mode="simple",
    )


@dsl.pipeline(
    name="Research LLM-Judge Evaluation Pipeline",
    description="Evaluate deep research agent with LLM-as-Judge scorers",
)
def llm_judge_eval_pipeline(
    mlflow_tracking_uri: str,
    llm_base_url: str,
    mlflow_workspace: str = "",
    llm_model: str = "granite-3.3-8b-instruct",
    mlflow_experiment_name: str = "deep-research-harness",
    dataset_name: str = "deep_research_eval_llm_judge",
    llm_secret_name: str = "llm-credentials",
):
    setup_task = setup_mlflow_op(
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
        mlflow_workspace=mlflow_workspace,
    )
    dataset_task = create_dataset_op(
        mlflow_tracking_uri=mlflow_tracking_uri,
        experiment_name=setup_task.output,
        dataset_name=dataset_name,
        mlflow_workspace=mlflow_workspace,
    )
    eval_task = run_eval_op(
        mlflow_tracking_uri=mlflow_tracking_uri,
        experiment_name=dataset_task.outputs["experiment_name"],
        dataset_id=dataset_task.outputs["dataset_id"],
        mode="llm-judge",
        llm_base_url=llm_base_url,
        llm_model=llm_model,
        mlflow_workspace=mlflow_workspace,
    )
    kubernetes.use_secret_as_env(
        eval_task,
        secret_name=llm_secret_name,
        secret_key_to_env={"OPENAI_API_KEY": "OPENAI_API_KEY"},
    )
    report_results_op(
        metrics=eval_task.output,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mode="llm-judge",
    )


print("\u2705 Pipelines defined (simple + llm-judge)")

## 5. Compile Pipelines to YAML

Compile the pipeline definitions to YAML files that can be uploaded to
RHOAI Data Science Pipelines.

In [ ]:
from kfp import compiler
from pathlib import Path

output_dir = Path("pipelines_gen")
output_dir.mkdir(exist_ok=True)

simple_path = output_dir / "simple-eval-pipeline.yaml"
compiler.Compiler().compile(
    pipeline_func=simple_eval_pipeline,
    package_path=str(simple_path),
)
print(f"\u2705 Simple pipeline: {simple_path}")

judge_path = output_dir / "llm-judge-eval-pipeline.yaml"
compiler.Compiler().compile(
    pipeline_func=llm_judge_eval_pipeline,
    package_path=str(judge_path),
)
print(f"\u2705 LLM-Judge pipeline: {judge_path}")

## 6. Upload and Run on RHOAI

Upload the compiled pipeline YAML to RHOAI Data Science Pipelines.

**Option A: RHOAI Dashboard (UI)**
1. Open the RHOAI Dashboard
2. Navigate to **Data Science Pipelines** > **Pipelines**
3. Click **Import Pipeline** and upload the YAML file
4. Click **Create Run**, fill in the parameters, and start

**Option B: KFP Python Client (programmatic)**

In [ ]:
# Uncomment and configure to run programmatically:

# from kfp.client import Client
#
# dspa_route = subprocess.run(
#     ["oc", "get", "route", "ds-pipeline-dspa", "-n", ns, "-o",
#      "jsonpath={.spec.host}"],
#     capture_output=True, text=True,
# ).stdout.strip()
#
# token = subprocess.run(
#     ["oc", "whoami", "-t"], capture_output=True, text=True
# ).stdout.strip()
#
# client = Client(
#     host=f"https://{dspa_route}",
#     existing_token=token,
#     ssl_ca_cert=False,
# )
#
# run = client.create_run_from_pipeline_package(
#     str(simple_path),
#     arguments={
#         "mlflow_tracking_uri": mlflow_uri,
#         "mlflow_experiment_name": experiment_name,
#     },
# )
# print(f"Pipeline run started: {run.run_id}")

print("\u2705 Pipeline ready \u2014 upload the YAML to RHOAI or uncomment the code above")

## 7. Required Kubernetes Secrets

The pipeline steps authenticate using Kubernetes secrets. Create these before
running the LLM-Judge pipeline:

```yaml
# mlflow-credentials (auto-detected from ServiceAccount token on RHOAI)
apiVersion: v1
kind: Secret
metadata:
  name: mlflow-credentials
stringData:
  MLFLOW_TRACKING_TOKEN: <your-token>

# llm-credentials (for LLM-judge mode)
apiVersion: v1
kind: Secret
metadata:
  name: llm-credentials
stringData:
  OPENAI_API_KEY: <your-api-key>
```

## 8. Summary

In [ ]:
print("\u2705 Evaluation pipeline walkthrough complete")
print()
print("What we covered:")
print("  1. Defined 4 pipeline components (setup, dataset, eval, report)")
print("  2. Created simple and LLM-judge pipeline variants")
print("  3. Compiled pipelines to YAML")
print("  4. Learned how to upload and run on RHOAI Data Science Pipelines")
print()
print("Generated files:")
print(f"  {simple_path}")
print(f"  {judge_path}")
print()
print("\u2705 Phase 6 complete \u2014 Custom Deep Research evaluation on RHOAI")